This chapter focuses on recommendation systems. We will start with a simple recommendation system that recommends the most popular items and then move onto ones that are more complex and recommend personalized items.

In [5]:
#Import a new datafile and read it into Python
import pandas as pd
import numpy as np
interaction=pd.read_csv('https://bradfordtuckfield.com/purchasehistory1.csv')
interaction.set_index("Unnamed: 0", inplace = True)
print(interaction)

            user1  user2  user3  user4  user5
Unnamed: 0                                   
item1           1      1      0      1      1
item2           1      0      1      1      0
item3           1      1      0      1      1
item4           1      0      1      0      1
item5           1      1      0      0      1


In [6]:
#Sum the purchase counts for each item and create a list with the most popular items.
interaction_withcounts=interaction.copy()
interaction_withcounts.loc[:,'counts']=interaction_withcounts.sum(axis=1)
interaction_withcounts=interaction_withcounts.sort_values(by='counts',ascending=False)
print(list(interaction_withcounts.index))

['item1', 'item3', 'item2', 'item4', 'item5']


In [7]:
#This uses all of the previous code to make a function that is a popularity based recommendation system.
def popularity_based(interaction):
  interaction_withcounts=interaction.copy()
  interaction_withcounts.loc[:,'counts']=interaction_withcounts.sum(axis=1)
  sorted = interaction_withcounts.sort_values(by='counts',ascending=False)
  most_popular=list(sorted.index)
  return(most_popular)

In [8]:
#Same output as the previous
print(popularity_based(interaction))

['item1', 'item3', 'item2', 'item4', 'item5']


Now we move onto the next recommendation system, collaborative filtering. This method recommends items to users based on what items are most commonly bought together.

We can use the concept of cosine similarity to find the similarity of different items.

In [9]:
#Print the purchase history of item 1.
print(list(interaction.loc['item1',:]))

[1, 1, 0, 1, 1]


In [10]:
#Create a function that can calculate the dot product of two vectors of the same length.
def dot_product(vector1,vector2):
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])
  return(thedotproduct)

In [11]:
#Create a function to calculate the norm of any vector.
def vector_norm(vector):
  thenorm=np.sqrt(dot_product(vector,vector))
  return(thenorm)

In [12]:
#Create a function to find the cosine similarity using the two previous functions.
def cosine_similarity(vector1,vector2):
  thedotproduct=dot_product(vector1,vector2)
  thecosine=thedotproduct/(vector_norm(vector1)*vector_norm(vector2))
  thecosine=np.round(thecosine,4)
  return(thecosine)

In [14]:
#We can calculate the cosine similarity between items 1 and 3 to get an output of 1. This is because they are identical vectors.
import numpy as np
item1=interaction.loc['item1',:]
item3=interaction.loc['item3',:]
print(cosine_similarity(item1,item3))

1.0


/tmp/ipython-input-1750573894.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [16]:
#For items 2 and 5, we get a result of 0.3333. This means that the two items are quite different.
item2=list(interaction.loc['item2',:])
item5=list(interaction.loc['item5',:])
print(cosine_similarity(item2,item5))

0.3333


Now we can start to actually implement item-based collaborative filtering.

In [19]:
#Define the vectors needed for the calculations
ouritem='item1'
otherrows=[rowname for rowname in interaction.index if rowname!=ouritem]
otheritems=interaction.loc[otherrows,:]
theitem=interaction.loc[ouritem,:]

In [21]:
#Calculate how similar each item is to the selected item and recommend items based on which are most similar.
similarities=[]
for items in otheritems.index:
  similarities.append(cosine_similarity(theitem,otheritems.loc[items,:]))
otheritems['similarities']=similarities
recommendations = list(otheritems.sort_values(by='similarities',ascending=False).index)

/tmp/ipython-input-1750573894.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [23]:
#Print the recommendations sorted from most relevant to least relevant.
print(recommendations)

['item3', 'item5', 'item2', 'item4']


In [25]:
#Create a collaborative filtering function that combines all of the previous code
def get_item_recommendations(interaction,itemname):
  otherrows=[rowname for rowname in interaction.index if rowname!=itemname]
  otheritems=interaction.loc[otherrows,:]
  theitem=list(interaction.loc[itemname,:])
  similarities=[]
  for items in otheritems.index:
    similarities.append(cosine_similarity(theitem,list(otheritems.loc[items,:])))
  otheritems['similarities']=similarities
  return list(otheritems.sort_values(by='similarities',ascending=False).index)

In [27]:
#This lets you see the order of recommendations for people interested in item 1.
get_item_recommendations(interaction,'item1')

['item3', 'item5', 'item2', 'item4']

Now we move onto user-based collaborative filtering, which recommends items based on what similar people have bought.

In [30]:
#Calculate the cosine similarity between users 2 and 5 just like we did with item-based collaborative filtering.
#We find an output of 0.866, meaning that they are closely related.
user2=interaction.loc[:,'user2']
user5=interaction.loc[:,'user5']
print(cosine_similarity(user2,user5))

0.866


/tmp/ipython-input-1750573894.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [31]:
#Do the same with users 3 and 5. We get an output of 0.3536, which means they are not very related.
user3=interaction.loc[:,'user3']
user5=interaction.loc[:,'user5']
print(cosine_similarity(user3,user5))

0.3536


/tmp/ipython-input-1750573894.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [32]:
#Create a function that calculates the most similar customers to a given person.
def get_similar_users(interaction,username):
  othercolumns=[columnname for columnname in interaction.columns if columnname!=username]
  otherusers=interaction[othercolumns]
  theuser=list(interaction[username])
  similarities=[]
  for users in otherusers.columns:
    similarities.append(cosine_similarity(theuser,list(otherusers.loc[:,users])))
  otherusers.loc['similarities',:]=similarities
  return list(otherusers.sort_values(by='similarities',axis=1,ascending=False).columns)

In [33]:
#Create a function that utilizes the previous code to implement user-based collaborative filtering.
def get_user_recommendations(interaction,username):
  similar_users=get_similar_users(interaction,username)
  purchase_history=interaction[similar_users[0]]
  purchased=list(purchase_history.loc[purchase_history==1].index)
  purchased2=list(interaction.loc[interaction[username]==1,:].index)
  recs=sorted(list(set(purchased) - set(purchased2)))
  return(recs)

In [35]:
#Run the code to get a recommended item using user-based collaborative filtering.
#We get item 4 because User 5 is the most similar to user 2. User 5 has purchased item 4, but user 2 has not.
get_user_recommendations (interaction,'user2')

/tmp/ipython-input-3143165071.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  otherusers.loc['similarities',:]=similarities


['item4']

Now we will work on a case study where we implement a recommendation system on real users and items.

In [43]:
#Load in the new data.
import pandas as pd
lastfm = pd.read_csv("https://bradfordtuckfield.com/lastfm-matrix-germany.csv")
print(lastfm.head())

   user  a perfect circle  abba  ac/dc  adam green  aerosmith  afi  air  \
0     1                 0     0      0           0          0    0    0   
1    33                 0     0      0           1          0    0    0   
2    42                 0     0      0           0          0    0    0   
3    51                 0     0      0           0          0    0    0   
4    62                 0     0      0           0          0    0    0   

   alanis morissette  alexisonfire  ...  timbaland  tom waits  tool  \
0                  0             0  ...          0          0     0   
1                  0             0  ...          0          0     0   
2                  0             0  ...          0          0     0   
3                  0             0  ...          0          0     0   
4                  0             0  ...          0          0     0   

   tori amos  travis  trivium  u2  underoath  volbeat  yann tiersen  
0          0       0        0   0          0        

In [44]:
#Drop the first column since it is unnecessary.
lastfm.drop(['user'],axis=1,inplace=True)

In [45]:
#Transpose the data to make it compatible with our functions.
lastfmt=lastfm.T

In [47]:
#Check the number of rows and columns.
#We have 1257 users and 285 artists.
print(lastfmt.shape)

(285, 1257)


In [52]:
#You can perform item-based collective filtering on the data via the function we already made.
#We get recommendations for a user interested in 'abba'
get_item_recommendations(lastfmt,'abba')[0:10]

['madonna',
 'robbie williams',
 'elvis presley',
 'michael jackson',
 'queen',
 'the beatles',
 'kelly clarkson',
 'groove coverage',
 'duffy',
 'mika']

In [53]:
#We can do user-based collective filtering as well by calling our premade function.
#We get recommednations for user 0 based on the most similar other user
print(get_user_recommendations(lastfmt,0)[0:3])

/tmp/ipython-input-94138093.py:4: RuntimeWarning: invalid value encountered in scalar divide
  thecosine=thedotproduct/(vector_norm(vector1)*vector_norm(vector2))


['billy talent', 'bob marley', 'die toten hosen']


/tmp/ipython-input-3143165071.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  otherusers.loc['similarities',:]=similarities
